In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def build_big_wide_table(
    csv_path: str,
    value_col: str,
    out_tex: str,
    caption: str,
    label: str,
    digits: int = 3,
    diagram_order=("flowchart", "graph", "stateDiagram"),
    difficulty_order=("Easy", "Moderate", "Hard"),
):
    df = pd.read_csv(csv_path)

    # ---- normalize model ----
    def norm_model(m):
        m = str(m).strip().lower()
        if m in {"gpt-4.1", "4.1", "gpt4.1"}:
            return "GPT-4.1"
        if "o4-mini" in m:
            return "GPT-o4-mini"
        return str(m).strip()

    # ---- normalize tier ----
    def norm_tier(row):
        tier = str(row["tier"]).strip()
        model = str(row["model"]).strip().lower()
        if tier != "v3":
            return tier
        if "reasoning_medium" in model:
            return "v3-med"
        if "reasoning_high" in model:
            return "v3-high"
        return "v3"

    df["model_norm"] = df["model"].apply(norm_model)
    df["tier_norm"] = df.apply(norm_tier, axis=1)

    # Pivot
    wide = df.pivot_table(
        index=["diagram_type", "difficulty"],
        columns=["model_norm", "tier_norm"],
        values=value_col,
        aggfunc="mean",
    )

    gpt_tiers = [t for t in ["base", "v1", "v2", "v3"] if ("GPT-4.1", t) in wide.columns]
    o4_tiers = [t for t in ["base", "v1", "v2", "v3-med", "v3-high"] if ("GPT-o4-mini", t) in wide.columns]

    diag_rank = {d: i for i, d in enumerate(diagram_order)}
    diff_rank = {d: i for i, d in enumerate(difficulty_order)}
    idx_sorted = sorted(list(wide.index), key=lambda x: (diag_rank.get(x[0], 999), diff_rank.get(x[1], 999)))

    def fmt(x):
        if pd.isna(x):
            return "—"
        return f"{x:.{digits}f}"

    def pretty_diag(d):
        return {"flowchart": "Flowchart", "graph": "Graph", "stateDiagram": "State Diagram"}.get(d, d)

    ncols = 2 + len(gpt_tiers) + len(o4_tiers)
    colspec = "ll" + "c" * (ncols - 2)

    lines = []
    lines += [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\resizebox{\textwidth}{!}{%",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
        rf"\textbf{{Diagram}} & \textbf{{Difficulty}} & "
        rf"\multicolumn{{{len(gpt_tiers)}}}{{c}}{{\textbf{{GPT-4.1}}}} & "
        rf"\multicolumn{{{len(o4_tiers)}}}{{c}}{{\textbf{{GPT-o4-mini}}}} \\",
        rf"\cmidrule(lr){{3-{2 + len(gpt_tiers)}}}\cmidrule(lr){{{3 + len(gpt_tiers)}-{2 + len(gpt_tiers) + len(o4_tiers)}}}",
        r"& & " + " & ".join([rf"\textbf{{{t}}}" for t in (gpt_tiers + o4_tiers)]) + r" \\",
        r"\midrule",
    ]

    by_diag = {}
    for diag, diff in idx_sorted:
        by_diag.setdefault(diag, []).append(diff)

    for diag in diagram_order:
        if diag not in by_diag:
            continue
        diffs = by_diag[diag]
        for i, diff in enumerate(diffs):
            row = wide.loc[(diag, diff)]

            gvals = [row[("GPT-4.1", t)] for t in gpt_tiers]
            ovals = [row[("GPT-o4-mini", t)] for t in o4_tiers]
            gmax = np.nanmax(gvals) if len(gvals) else np.nan
            omax = np.nanmax(ovals) if len(ovals) else np.nan

            def cell(v, vmax):
                if pd.isna(v):
                    return "—"
                if not pd.isna(vmax) and np.isclose(v, vmax):
                    return rf"\textbf{{{fmt(v)}}}"
                return fmt(v)

            g_cells = [cell(row[("GPT-4.1", t)], gmax) for t in gpt_tiers]
            o_cells = [cell(row[("GPT-o4-mini", t)], omax) for t in o4_tiers]

            diag_cell = rf"\multirow{{{len(diffs)}}}{{*}}{{\textbf{{{pretty_diag(diag)}}}}}" if i == 0 else ""
            lines.append(diag_cell + " & " + diff + " & " + " & ".join(g_cells + o_cells) + r" \\")
        lines.append(r"\addlinespace")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"}",  # closes resizebox
        r"\end{table}",
    ]

    Path(out_tex).write_text("\n".join(lines), encoding="utf-8")
    print("Wrote:", out_tex)

In [ ]:
build_big_wide_table(
    csv_path="WLSimilarity_merged.csv",
    value_col="avg_WLSimilarity",
    out_tex="tables/Average_WL_similarity_wide.tex",
    caption="Average WL similarity by diagram type and difficulty. For each row, the best prompt tier within each model is bolded.",
    label="tab:wl_5_1",
)

In [ ]:
build_big_wide_table(
    csv_path="DirectedSpectralSimilarity_merged.csv",
    value_col="avg_DirectedSpectralSimilarity",
    out_tex="tables/Average_Directed_Spectral_Similarity_wide.tex",
    caption="Average Directed Spectral similarity by diagram type and difficulty. For each row, the best prompt tier within each model is bolded.",
    label="tab:spectral_5_1",
)

In [ ]:
build_big_wide_table(
    csv_path="DirectedErrorEvaluator_merged.csv",
    value_col="avg_f1_score",
    out_tex="tables/Average_F1_wide.tex",
    caption="Average structural F1 by diagram type and difficulty. For each row, the best prompt tier within each model is bolded.",
    label="tab:f1_wide",
)

In [ ]:
import polars as pl

In [ ]:
pl.read_parquet("results/results_DirectedErrorEvaluator_base_gpt-4.1.parquet").columns

In [ ]:
import re
from collections.abc import Iterable, Sequence
from typing import Optional

import polars as pl


def merge_per_image_results(
    results_dir: str,
    metric_prefix: str,
    keep_cols: Sequence[str],
    out_parquet: str,
    out_csv: str | None = None,
    file_glob: str | None = None,
    allowed_tiers: Iterable[str] = ("base", "v1", "v2", "v3"),
):
    """
    Merge per-image results for ANY metric into one dataset.

    Expected filename pattern:
        results_<metric_prefix>_<tier>_<model>.parquet

    Example:
        results_DirectedErrorEvaluator_base_gpt-4.1.parquet

    Parameters
    ----------
    results_dir : str
        Folder containing the parquet result files.
    metric_prefix : str
        Metric name in the filename (e.g. "WLSimilarity", "DirectedSpectralSimilarity", "DirectedErrorEvaluator").
    keep_cols : Sequence[str]
        Columns to keep from each parquet (must exist).
        Typical: ["diagram_type","difficulty","image_filename","<metric_value_col>"]
    out_parquet : str
        Output parquet path.
    out_csv : Optional[str]
        Output csv path (optional).
    file_glob : Optional[str]
        Override file pattern. Default uses f"results_{metric_prefix}_*.parquet".
    allowed_tiers : Iterable[str]
        Allowed tier strings in filenames.

    Returns
    -------
    polars.DataFrame
    """

    results_path = Path(results_dir)
    glob_pattern = file_glob or f"results_{metric_prefix}_*.parquet"
    files = sorted(results_path.glob(glob_pattern))

    print(f"[{metric_prefix}] Found {len(files)} files in {results_path}")

    tier_alt = "|".join(map(re.escape, allowed_tiers))
    pattern = re.compile(rf"^results_{re.escape(metric_prefix)}_(?P<tier>{tier_alt})_(?P<model>.+)\.parquet$")

    dfs = []
    for f in files:
        m = pattern.match(f.name)
        if not m:
            print("  Skipping (pattern mismatch):", f.name)
            continue

        tier = m.group("tier")
        model = m.group("model")

        df = (
            pl.read_parquet(f)
            .select(list(keep_cols))
            .with_columns(
                [
                    pl.lit(tier).alias("tier"),
                    pl.lit(model).alias("model"),
                    pl.lit(metric_prefix).alias("metric"),
                    pl.lit(f.name).alias("source_file"),
                ]
            )
        )
        dfs.append(df)

    if not dfs:
        raise RuntimeError(f"[{metric_prefix}] No files matched. Check results_dir, glob, and filename pattern.")

    merged = pl.concat(dfs, how="vertical")

    # Helpful sanity checks
    if "image_filename" in merged.columns:
        n_images = merged.select(pl.col("image_filename").n_unique()).item()
        print(f"[{metric_prefix}] Unique images: {n_images}")

    print(f"[{metric_prefix}] Total rows: {merged.height}")
    # Expect ~ 900 * (#configs). If you have 9 configs => 8100.

    merged.write_parquet(out_parquet)
    if out_csv:
        merged.write_csv(out_csv)

    print(f"[{metric_prefix}] Wrote: {out_parquet}")
    if out_csv:
        print(f"[{metric_prefix}] Wrote: {out_csv}")

    return merged

In [ ]:
merge_per_image_results(
    results_dir="results",
    metric_prefix="WLSimilarity",
    keep_cols=["diagram_type", "difficulty", "image_filename", "WLSimilarity"],
    out_parquet="merged_all_config/WLSimilarity_merged_per_image.parquet",
    out_csv="merged_all_config/WLSimilarity_merged_per_image.csv",
)

In [ ]:
def build_table_base_vs_best(
    merged_stats_csv: str,
    value_col: str,
    out_tex: str,
    caption: str,
    label: str,
    digits: int = 3,
):
    """
    Table 5.2 (MACRO):
    Average metric for base prompting vs best-performing tier.
    Uses merged stats file (mean-of-means across diagram × difficulty cells).
    """

    df = pd.read_csv(merged_stats_csv)

    # Normalize model
    def norm_model(m):
        m = str(m).strip().lower()
        if m in {"gpt-4.1", "4.1", "gpt4.1"}:
            return "GPT-4.1"
        if "o4-mini" in m:
            return "o4-mini"
        return str(m).strip()

    # Normalize tier
    def norm_tier(row):
        t = str(row["tier"]).strip()
        m = str(row["model"]).strip().lower()
        if t != "v3":
            return t
        if "reasoning_medium" in m:
            return "v3-med"
        if "reasoning_high" in m:
            return "v3-high"
        return "v3"

    df["model_norm"] = df["model"].apply(norm_model)
    df["tier_norm"] = df.apply(norm_tier, axis=1)

    # Macro mean per model x tier
    summary = (
        df.groupby(["model_norm", "tier_norm"], as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: "macro_mean"})
    )

    rows = []
    for model in ["GPT-4.1", "o4-mini"]:
        sub = summary[summary["model_norm"] == model]
        if sub.empty:
            continue

        base_val = float(sub[sub["tier_norm"] == "base"]["macro_mean"].iloc[0])
        best_row = sub.loc[sub["macro_mean"].idxmax()]
        best_tier = best_row["tier_norm"]
        best_val = float(best_row["macro_mean"])
        delta = best_val - base_val

        rows.append((model, base_val, best_tier, best_val, delta))

    def fmt(x):
        return f"{x:.{digits}f}"

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lcccc}",
        r"\toprule",
        r"\textbf{Model} & \textbf{Avg (Base)} & \textbf{Best Tier} & \textbf{Avg (Best)} & \textbf{$\Delta$} \\",
        r"\midrule",
    ]

    for model, base_val, best_tier, best_val, delta in rows:
        lines.append(f"{model} & {fmt(base_val)} & {best_tier} & {fmt(best_val)} & {fmt(delta)} \\\\")

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    Path(out_tex).write_text("\n".join(lines))
    print("Wrote:", out_tex)

In [ ]:
def build_table_improvement_by_difficulty(
    merged_stats_csv: str,
    value_col: str,
    out_tex: str,
    caption: str,
    label: str,
    digits: int = 3,
    difficulty_order=("Easy", "Moderate", "Hard"),
):
    """
    Table 5.3 (MACRO):
    Improvement per difficulty level.
    Uses merged stats (equal-weight across diagram types).
    """

    df = pd.read_csv(merged_stats_csv)

    def norm_model(m):
        m = str(m).strip().lower()
        if m in {"gpt-4.1", "4.1", "gpt4.1"}:
            return "GPT-4.1"
        if "o4-mini" in m:
            return "o4-mini"
        return str(m).strip()

    def norm_tier(row):
        t = str(row["tier"]).strip()
        m = str(row["model"]).strip().lower()
        if t != "v3":
            return t
        if "reasoning_medium" in m:
            return "v3-med"
        if "reasoning_high" in m:
            return "v3-high"
        return "v3"

    df["model_norm"] = df["model"].apply(norm_model)
    df["tier_norm"] = df.apply(norm_tier, axis=1)

    rows = []

    for model in ["GPT-4.1", "o4-mini"]:
        for diff in difficulty_order:
            sub = df[(df["model_norm"] == model) & (df["difficulty"] == diff)]
            if sub.empty:
                continue

            # Macro mean within difficulty (equal weight across diagram types)
            tier_means = sub.groupby("tier_norm")[value_col].mean()

            base_val = float(tier_means["base"])
            best_tier = tier_means.idxmax()
            best_val = float(tier_means.max())
            delta = best_val - base_val
            rel = (delta / base_val) * 100.0 if base_val != 0 else 0.0

            rows.append((model, diff, base_val, best_val, delta, rel))

    def fmt(x):
        return f"{x:.{digits}f}"

    def fmt_delta(x):
        sign = "+" if x >= 0 else ""
        return f"{sign}{x:.{digits}f}"

    def fmt_pct(x):
        sign = "+" if x >= 0 else ""
        return f"{sign}{x:.1f}\\%"

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{llcccc}",
        r"\toprule",
        r"\textbf{Model} & \textbf{Difficulty} & \textbf{Base} & \textbf{Best} & \textbf{$\Delta$} & \textbf{Relative $\Delta$} \\",
        r"\midrule",
    ]

    for model, diff, base_val, best_val, delta, rel in rows:
        lines.append(f"{model} & {diff} & {fmt(base_val)} & {fmt(best_val)} & {fmt_delta(delta)} & {fmt_pct(rel)} \\\\")

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    Path(out_tex).write_text("\n".join(lines))
    print("Wrote:", out_tex)

In [ ]:
# ---- WL Table 5.2 ----
build_table_base_vs_best(
    merged_stats_csv="DirectedErrorEvaluator_merged.csv",
    value_col="avg_f1_score",
    out_tex="tables/f1_base_vs_best.tex",
    caption="Average F1 score for base prompting versus best-performing prompt tier.",
    label="tab:f1_base_vs_best",
)

# ---- WL Table 5.3 ----
build_table_improvement_by_difficulty(
    merged_stats_csv="DirectedErrorEvaluator_merged.csv",
    value_col="avg_f1_score",
    out_tex="tables/f1_improve_by_difficulty.tex",
    caption="Average F1 score improvement per difficulty level.",
    label="tab:f1_improve_by_difficulty",
)

In [ ]:
# ---- WL Table 5.2 ----
build_table_base_vs_best(
    merged_stats_csv="DirectedSpectralSimilarity_merged.csv",
    value_col="avg_DirectedSpectralSimilarity",
    out_tex="tables/spectral_similarity_base_vs_best.tex",
    caption="Average Directed Spectral Similarity for base prompting versus best-performing prompt tier.",
    label="tab:pectral_similarity_base_vs_best",
)

# ---- WL Table 5.3 ----
build_table_improvement_by_difficulty(
    merged_stats_csv="DirectedSpectralSimilarity_merged.csv",
    value_col="avg_DirectedSpectralSimilarity",
    out_tex="tables/spectral_similarity_improve_by_difficulty.tex",
    caption="Average Directed Spectral Similarity improvement per difficulty level.",
    label="tab:spectral_similarity_improve_by_difficulty",
)

In [ ]:
def build_table_improvement_by_diagram(
    merged_stats_csv: str,
    value_col: str,
    out_tex: str,
    caption: str,
    label: str,
    digits: int = 3,
    pct_digits: int = 1,
    diagram_order=("flowchart", "graph", "stateDiagram"),
):
    """
    Table 5.4 (MACRO): Improvement (Δ) by diagram type, using merged stats (mean-of-means).
    - Groups rows by model using \\multirow{3}{*}{...}
    - Adds \\midrule separator line between model blocks
    - Reports Base, Best, signed Δ, signed Relative Δ

    Requires LaTeX packages:
      \\usepackage{booktabs}
      \\usepackage{multirow}

    Expected columns in merged_stats_csv:
      - diagram_type, tier, model, and value_col
    """

    df = pd.read_csv(merged_stats_csv)

    # Normalize model for display
    def norm_model(m):
        m = str(m).strip().lower()
        if m in {"gpt-4.1", "4.1", "gpt4.1"}:
            return "GPT-4.1"
        if "o4-mini" in m:
            return "o4-mini"
        return str(m).strip()

    # Normalize tier (split v3 into v3-med/v3-high based on model string)
    def norm_tier(row):
        t = str(row["tier"]).strip()
        m = str(row["model"]).strip().lower()
        if t != "v3":
            return t
        if "reasoning_medium" in m:
            return "v3-med"
        if "reasoning_high" in m:
            return "v3-high"
        return "v3"

    df["model_norm"] = df["model"].apply(norm_model)
    df["tier_norm"] = df.apply(norm_tier, axis=1)

    def pretty_diag(d):
        return {"flowchart": "Flowchart", "graph": "Graph", "stateDiagram": "State Diagram"}.get(d, d)

    models = ["GPT-4.1", "o4-mini"]

    # For each (model, diagram_type):
    # macro mean across difficulties already in merged stats -> just mean over rows for that diagram type
    rows = []  # (model, diagram_type, base, best, delta, rel, best_tier)
    for model in models:
        for diag in diagram_order:
            sub = df[(df["model_norm"] == model) & (df["diagram_type"] == diag)]
            if sub.empty:
                continue

            tier_means = sub.groupby("tier_norm")[value_col].mean()  # macro across difficulties (equal weight)

            if "base" not in tier_means.index:
                raise ValueError(f"Missing base tier for model={model}, diagram_type={diag}")

            base = float(tier_means["base"])
            best_tier = tier_means.idxmax()
            best = float(tier_means.max())
            delta = best - base
            rel = (delta / base) * 100.0 if base != 0 else 0.0

            rows.append((model, pretty_diag(diag), base, best, delta, rel, best_tier))

    def fmt(x):
        return f"{x:.{digits}f}"

    def fmt_signed(x):
        return f"{'+' if x >= 0 else ''}{x:.{digits}f}"

    def fmt_pct_signed(x):
        return f"{'+' if x >= 0 else ''}{x:.{pct_digits}f}\\%"

    # Organize rows by model, keep diagram order
    rows_by_model = {m: [] for m in models}
    for r in rows:
        rows_by_model[r[0]].append(r)

    # sort each block by diagram order
    order_map = {pretty_diag(d): i for i, d in enumerate(diagram_order)}
    for m in models:
        rows_by_model[m].sort(key=lambda x: order_map.get(x[1], 999))

    # LaTeX
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{llccccc}",
        r"\toprule",
        r"\textbf{Model} & \textbf{Diagram Type} & \textbf{Base} & \textbf{Best} & \textbf{$\Delta$} & \textbf{Relative $\Delta$} & \textbf{Best Tier} \\",
        r"\midrule",
    ]

    first_block = True
    for model in models:
        block = rows_by_model.get(model, [])
        if not block:
            continue

        if not first_block:
            lines.append(r"\midrule")  # separator between model blocks

        # multirow on first line of block
        for i, (m, diag, base, best, delta, rel, best_tier) in enumerate(block):
            model_cell = rf"\multirow{{{len(block)}}}{{*}}{{{m}}}" if i == 0 else ""
            lines.append(
                f"{model_cell} & {diag} & {fmt(base)} & {fmt(best)} & {fmt_signed(delta)} & {fmt_pct_signed(rel)} & {best_tier} \\\\"
            )

        first_block = False

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    Path(out_tex).write_text("\n".join(lines), encoding="utf-8")
    print("Wrote:", out_tex)

In [ ]:
build_table_improvement_by_diagram(
    merged_stats_csv="WLSimilarity_merged.csv",
    value_col="avg_WLSimilarity",
    out_tex="tables/wl_improve_by_diagram.tex",
    caption="Average WL similarity improvement ($\\Delta$) by diagram type (macro-averaged across difficulty levels).",
    label="tab:wl_improve_by_diagram",
)

In [ ]:
build_table_improvement_by_diagram(
    merged_stats_csv="DirectedSpectralSimilarity_merged.csv",
    value_col="avg_DirectedSpectralSimilarity",
    out_tex="tables/spectral_improve_by_diagram.tex",
    caption="Average Directed Spectral similarity improvement ($\\Delta$) by diagram type (macro-averaged across difficulty levels).",
    label="tab:spectral_improve_by_diagram",
)

In [ ]:
build_table_improvement_by_diagram(
    merged_stats_csv="DirectedErrorEvaluator_merged.csv",
    value_col="avg_f1_score",
    out_tex="tables/f1_improve_by_diagram.tex",
    caption="Average structural F1 improvement ($\\Delta$) by diagram type (macro-averaged across difficulty levels).",
    label="tab:f1_improve_by_diagram",
)